In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("D:\AB testing project\marketing_AB.csv")

In [4]:
df.shape

(588101, 7)

In [5]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [7]:
df.columns

Index(['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads',
       'most ads day', 'most ads hour'],
      dtype='object')

In [8]:
df["test group"].value_counts()

test group
ad     564577
psa     23524
Name: count, dtype: int64

In [9]:
df.groupby("test group")["converted"].mean()

test group
ad     0.025547
psa    0.017854
Name: converted, dtype: float64

In [10]:
df.isnull().sum()

Unnamed: 0       0
user id          0
test group       0
converted        0
total ads        0
most ads day     0
most ads hour    0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(0)

In [15]:
df["test group"].unique()

array(['ad', 'psa'], dtype=object)

In [19]:
df["converted"].unique()

array([False,  True])

In [20]:
df.groupby("test group")["converted"].agg(
    total_users="count",
    conversions="sum"
)

,total_users,conversions
test group,,
ad,564577,14423
psa,23524,420


In [21]:
conversion_rates = df.groupby("test group")["converted"].mean()

difference = conversion_rates["ad"] - conversion_rates["psa"]

print("Conversion rate difference:", difference)
print("Difference in percentage points:", difference * 100)

Conversion rate difference: 0.007692453192201517
Difference in percentage points: 0.7692453192201517


In [22]:
ad_rate = conversion_rates["ad"]
psa_rate = conversion_rates["psa"]

uplift = ((ad_rate - psa_rate) / psa_rate) * 100

print("Uplift:", uplift, "%")

Uplift: 43.085064022225836 %


In [23]:
from statsmodels.stats.proportion import proportions_ztest

In [24]:


conversions = [14423, 420]
users = [564577, 23524]

z_stat, p_value = proportions_ztest(conversions, users)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: 7.3700781265454145
P-value: 1.7052807161559727e-13


In [25]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    14423, 564577,
    420, 23524,
    method="wald"
)

print("95% CI - lower:", ci_low)
print("95% CI - upper:", ci_high)
print("95% CI in percentage points:", ci_low * 100, "to", ci_high * 100)

95% CI - lower: 0.005950932431611005
95% CI - upper: 0.00943397395279203
95% CI in percentage points: 0.5950932431611005 to 0.943397395279203


In [26]:
df = df.drop(columns=["Unnamed: 0"])

In [29]:
pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [35]:
from sqlalchemy import create_engine
username = "postgres"
password = ""
host = "localhost"
port = "5432"
database = "ab_testing"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = "testing"
df.to_sql(table_name, engine, if_exists = "replace", index = False)

print(f" data successfully loaded into table '{table_name}' in database '{database}'.")


 data successfully loaded into table 'testing' in database 'ab_testing'.
